# 02 — RAG Evaluation

Evaluate a toy Retrieval-Augmented Generation pipeline end to end:

* **Faithfulness** — is the answer grounded in the retrieved context?
* **Context Precision / Recall** — are the right documents retrieved?
* **NDCG / MRR** — ranking-quality of the retriever.

All judged metrics use an offline `FakeJudge`.

## 1. A toy RAG corpus and retriever

In [ ]:
CORPUS = [
    {"id": "d1", "text": "Python was created by Guido van Rossum in 1991."},
    {"id": "d2", "text": "Python emphasizes code readability."},
    {"id": "d3", "text": "Rust was first released in 2010 by Graydon Hoare."},
    {"id": "d4", "text": "Java was released by Sun Microsystems in 1995."},
]


def retrieve(query: str, k: int = 2):
    """Keyword retriever that returns the top-k documents containing query terms."""
    terms = set(query.lower().split())
    scored = []
    for doc in CORPUS:
        overlap = sum(1 for t in terms if t in doc["text"].lower())
        if overlap:
            scored.append((overlap, doc))
    scored.sort(key=lambda x: x[0], reverse=True)
    return [d for _, d in scored[:k]]


question = "Who created Python?"
retrieved = retrieve(question, k=2)
for doc in retrieved:
    print(doc)

## 2. Deterministic retrieval-quality metrics (no LLM needed)

In [ ]:
relevant_ids = {"d1", "d2"}
retrieved_ids = [d["id"] for d in retrieved]

hits = [1 if rid in relevant_ids else 0 for rid in retrieved_ids]
precision = sum(hits) / len(hits) if hits else 0.0
recall = sum(hits) / len(relevant_ids) if relevant_ids else 0.0
print("Precision@k:", round(precision, 3))
print("Recall@k   :", round(recall, 3))

## 3. Ranking-quality: NDCG and MRR

In [ ]:
import math


def dcg(gains):
    return sum(g / math.log2(i + 2) for i, g in enumerate(gains))


def ndcg(ranked_ids, relevant_ids):
    gains = [1.0 if rid in relevant_ids else 0.0 for rid in ranked_ids]
    ideal = sorted(gains, reverse=True)
    if not any(ideal):
        return 0.0
    return dcg(gains) / dcg(ideal)


def mrr(ranked_ids, relevant_ids):
    for i, rid in enumerate(ranked_ids, start=1):
        if rid in relevant_ids:
            return 1.0 / i
    return 0.0


print("NDCG:", round(ndcg(retrieved_ids, relevant_ids), 3))
print("MRR :", round(mrr(retrieved_ids, relevant_ids), 3))

## 4. Faithfulness of the generated answer (offline judge)

In [ ]:
from checkllm.judge import JudgeResponse


class FakeJudge:
    """Deterministic in-process judge used to keep this notebook offline.

    Real runs should swap this for ``OpenAIJudge`` / ``AnthropicJudge`` etc.
    """

    def __init__(self, score: float = 0.9, reasoning: str = "looks good") -> None:
        self._score = score
        self._reasoning = reasoning

    async def evaluate(self, prompt: str, system_prompt: str | None = None) -> JudgeResponse:
        return JudgeResponse(
            score=self._score,
            reasoning=self._reasoning,
            raw_output=str(self._score),
            cost=0.0,
        )


judge = FakeJudge(score=0.9, reasoning="offline stub")

In [ ]:
answer = "Python was created by Guido van Rossum in 1991."
context = "\n".join(d["text"] for d in retrieved)

prompt = (
    f"Question: {question}\nContext: {context}\nAnswer: {answer}\n"
    "Is the answer faithful to the context? Score 0-1."
)
result = await judge.evaluate(prompt=prompt)
print("faithfulness:", result.score, "|", result.reasoning)

## 5. Switch to a real provider

In [ ]:
# -------------------------------------------------------------------
# SWITCH TO A REAL PROVIDER
# -------------------------------------------------------------------
# Uncomment and replace the FakeJudge above once you have API keys:
#
# from checkllm.judge import OpenAIJudge
# judge = OpenAIJudge(api_key="sk-...", model="gpt-4o-mini")
#
# from checkllm.judge import AnthropicJudge
# judge = AnthropicJudge(api_key="...", model="claude-3-5-sonnet-20241022")
